In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML
from IPython.display import HTML as HTMLDisplay

processed_dir = Path("SourceCode-TKAnmedsos/data/processed")

folders = sorted(
    processed_dir.glob("cleaned_*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if not folders:
    raise FileNotFoundError(
        f"Tidak ditemukan folder cleaned_* di "
        f"{processed_dir.resolve()}"
    )

output_dir = folders[0]
print("Folder:", output_dir)

file_path = output_dir / "comments_analysis.parquet"

if not file_path.exists():
    raise FileNotFoundError(
        f"File tidak ditemukan: {file_path}"
    )

df = pd.read_parquet(file_path)

print(f"Jumlah komentar: {len(df):,}")

text_candidates = [
    "text_clean",
    "text_original",
    "text_with_emoji_names"
]

text_col = next(
    (col for col in text_candidates if col in df.columns),
    None
)

if text_col is None:
    raise KeyError(
        "Kolom teks komentar tidak ditemukan. "
        f"Kolom tersedia: {df.columns.tolist()}"
    )

comments = (
    df[[text_col]]
    .rename(columns={text_col: "Comment"})
    .reset_index(drop=True)
)

html_table = comments.to_html(
    index=False,
    escape=True,
    classes="comments-table",
    border=0
)

display(HTML(f"""
<style>
.comments-container {{
    height: 600px;
    overflow-y: auto;
    overflow-x: auto;
    border: 1px solid #bdbdbd;
    border-radius: 6px;
    width: 100%;
    background-color: #ffffff !important;
    color: #222222 !important;
}}

.comments-table {{
    border-collapse: collapse;
    width: 100%;
    table-layout: auto;
    background-color: #ffffff !important;
    color: #222222 !important;
    font-family: Arial, sans-serif;
    font-size: 14px;
}}

.comments-table th {{
    position: sticky;
    top: 0;
    z-index: 2;
    background-color: #e8e8e8 !important;
    color: #222222 !important;
    text-align: left;
    padding: 12px;
    border: 1px solid #cccccc;
}}

.comments-table td {{
    background-color: #ffffff !important;
    color: #222222 !important;
    padding: 10px 12px;
    border: 1px solid #dddddd;
    text-align: left;
    vertical-align: top;
    white-space: pre-wrap;
    overflow-wrap: anywhere;
    min-width: 300px;
    max-width: 1000px;
    line-height: 1.5;
}}

.comments-table tr:nth-child(even) td {{
    background-color: #f5f5f5 !important;
    color: #222222 !important;
}}

.comments-table tr:hover td {{
    background-color: #e3edff !important;
    color: #222222 !important;
}}
</style>

<div class="comments-container">
    {html_table}
</div>
"""))
print(f"Kolom teks yang ditampilkan: {text_col}")
print(f"Total baris yang ditampilkan: {len(comments):,}")